In [ ]:
import os, math, random, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_curve, roc_auc_score

# ----------------------------- CONFIG -----------------------------
DATA_ROOT  = "/kaggle/input/datasets/rohitbhargav/resnet-handwriting-11-06/train_final_images"  
                                                              
IMG_H, IMG_W = 128, 512          
IMAGES_PER_WRITER_PER_EPOCH = 80
EMBED_DIM  = 512
LAYERS     = [3, 4, 6, 3]        

BATCH_SIZE = 128                 
EPOCHS     = 80
BASE_LR    = 0.02
WEIGHT_DECAY = 1e-3
WARMUP_EPOCHS = 3
RESUME_PATH = None
ARC_S, ARC_M = 22.0, 0.30        
ARC_M_WARMUP_EPOCHS = 10         
SAVE_EVERY = 5
VAL_WRITER_FRAC = 0.15           
NUM_WORKERS = 4
SEED = 42
OUT_DIR = "/kaggle/working"
PAD_VALUE = 255                 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| GPUs:", torch.cuda.device_count())

In [ ]:
from pathlib import Path

root_dir = Path(r"/kaggle/input/models/rohitbhargav/resnet-handwriting-21/pytorch/default/1")

for item in root_dir.iterdir():
    if item.is_dir():
        print(f"[DIR ] {item.name}")
    elif item.is_file():
        print(f"[FILE] {item.name}")

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
# cudnn benchmark helps because every image is the same fixed 128x512 size
torch.backends.cudnn.benchmark = True

## 1. Split writers (open-set)

Disjoint writer sets. The ArcFace classifier is built **only** over the train writers.

In [ ]:
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

def scan_dataset(root):
    writers = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    w2imgs = {}
    for w in writers: 
        wd = os.path.join(root, w)
        imgs = [os.path.join(wd, f) for f in os.listdir(wd) if f.lower().endswith(IMG_EXTS)]
        if imgs:
            w2imgs[w] = imgs
    return w2imgs

w2imgs = scan_dataset(DATA_ROOT)
all_writers = sorted(w2imgs.keys())
print(f"Total writers: {len(all_writers)} | total images: {sum(len(v) for v in w2imgs.values())}")

rng = random.Random(SEED)
shuffled = all_writers[:]
rng.shuffle(shuffled)
n_val = int(round(len(shuffled) * VAL_WRITER_FRAC))
val_writers   = sorted(shuffled[:n_val])
train_writers = sorted(shuffled[n_val:])

# Contiguous label ids over TRAIN writers only (ArcFace #classes = len(train_writers))
writer_to_label = {w: i for i, w in enumerate(train_writers)}
NUM_CLASSES = len(train_writers)
print(f"train writers: {len(train_writers)} (= ArcFace classes) | val writers: {len(val_writers)}")

train_w2imgs = {w: w2imgs[w] for w in train_writers}
val_w2imgs   = {w: w2imgs[w] for w in val_writers}

## 2. Preprocessing + Dataset

`fit_image()` is the single source of truth for geometry and is used by **both** training and inference.
Resize to height 128 keeping aspect, then pad/crop width to 512.

The dataset re-samples **60 random images per writer every epoch** by calling `.resample()`.

In [ ]:
def fit_image(path, img_h=IMG_H, img_w=IMG_W, pad_value=PAD_VALUE):
    '''Load grayscale, resize to fixed height keeping aspect, pad/crop width -> (img_w, img_h) PIL 'L'.'''
    img = Image.open(path).convert("L")
    w, h = img.size
    new_w = max(1, int(round(w * img_h / h)))
    img = img.resize((new_w, img_h), Image.BILINEAR)
    if new_w > img_w:                     
        img = img.resize((img_w, img_h), Image.BILINEAR)
        new_w = img_w
    canvas = Image.new("L", (img_w, img_h), color=pad_value)
    canvas.paste(img, (0, 0))               
    return canvas


train_aug = T.RandomAffine(degrees=3, translate=(0.02, 0.04), scale=(0.95, 1.05),
                           fill=PAD_VALUE)
to_tensor = T.Compose([T.ToTensor(), T.Normalize([0.5], [0.5])])   

def make_tensor(pil_img, train):
    return to_tensor(pil_img)

In [ ]:
class WriterEpochDataset(Dataset):
    '''Each epoch holds `per_epoch` random images per writer. Call .resample() before every epoch.'''
    def __init__(self, w2imgs, writer_to_label, per_epoch, train, seed=0):
        self.w2imgs = w2imgs
        self.writer_to_label = writer_to_label
        self.per_epoch = per_epoch
        self.train = train
        self._rng = random.Random(seed)
        self.samples = []
        self.resample()

    def resample(self):
        s = []
        for w, imgs in self.w2imgs.items():
            if len(imgs) >= self.per_epoch:
                chosen = self._rng.sample(imgs, self.per_epoch)
            else:  # fallback; not needed for this dataset (min=100)
                chosen = imgs + self._rng.choices(imgs, k=self.per_epoch - len(imgs))
            label = self.writer_to_label[w]
            s.extend((p, label) for p in chosen)
        self._rng.shuffle(s)
        self.samples = s

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x = make_tensor(fit_image(path), train=self.train)
        return x, label

train_ds = WriterEpochDataset(train_w2imgs, writer_to_label,
                              IMAGES_PER_WRITER_PER_EPOCH, train=True, seed=SEED)
print("train samples/epoch:", len(train_ds), "(=", NUM_CLASSES, "writers x", IMAGES_PER_WRITER_PER_EPOCH, ")")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                          persistent_workers=(NUM_WORKERS > 0))

## 3. Model — ResNet-30 (1-channel) + ArcFace head

The backbone outputs an `EMBED_DIM` embedding. ArcFace normalizes both embedding and class weights and
applies the additive angular margin only during training.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample
    def forward(self, x):
        idn = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            idn = self.downsample(x)
        return F.relu(out + idn, inplace=True)

class ResNetBackbone(nn.Module):
    def __init__(self, layers, in_ch=1, embed_dim=512):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, 2, 3, bias=False),   # modified: 1-channel input
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        self.layer1 = self._make(64,  layers[0], 1)
        self.layer2 = self._make(128, layers[1], 2)
        self.layer3 = self._make(256, layers[2], 2)
        self.layer4 = self._make(512, layers[3], 2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(512, embed_dim)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def _make(self, planes, n, stride):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_planes, planes, 1, stride, bias=False),
                nn.BatchNorm2d(planes))
        blocks = [BasicBlock(self.in_planes, planes, stride, downsample)]
        self.in_planes = planes
        blocks += [BasicBlock(planes, planes) for _ in range(1, n)]
        return nn.Sequential(*blocks)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.gap(x).flatten(1)
        return self.fc(x)

In [ ]:
class ArcMarginProduct(nn.Module):
    '''ArcFace: additive angular margin. Returns scaled logits; apply CrossEntropy on top.'''
    def __init__(self, in_features, out_features, s=30.0, m=0.50, easy_margin=False):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_normal_(self.weight)
        self.easy_margin = easy_margin
        self._set_margin(m)
    def _set_margin(self, m):
        self.m = m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m
    def forward(self, emb, label):
        cosine = F.linear(F.normalize(emb), F.normalize(self.weight)).clamp(-1, 1)
        sine = torch.sqrt((1.0 - cosine**2).clamp(min=1e-9))
        phi = cosine * self.cos_m - sine * self.sin_m          # cos(theta + m)
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        onehot = torch.zeros_like(cosine)
        onehot.scatter_(1, label.view(-1, 1), 1.0)
        out = onehot * phi + (1.0 - onehot) * cosine
        return out * self.s

class HWVerifier(nn.Module):
    def __init__(self, layers, embed_dim, n_classes, s, m):
        super().__init__()
        self.backbone = ResNetBackbone(layers, in_ch=1, embed_dim=embed_dim)
        nn.Dropout(0.3)
        self.emb_bn   = nn.BatchNorm1d(embed_dim)   # standard ArcFace "BN-neck"
        self.arc      = ArcMarginProduct(embed_dim, n_classes, s=s, m=m)
    def embed(self, x):
        return F.normalize(self.emb_bn(self.backbone(x)))   # <-- use THIS at inference
    def forward(self, x, label):
        emb = self.emb_bn(self.backbone(x))
        return self.arc(emb, label)

model = HWVerifier(LAYERS, EMBED_DIM, NUM_CLASSES, ARC_S, ARC_M).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"params: {n_params:.1f}M | classes: {NUM_CLASSES}")
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print("Using DataParallel on", torch.cuda.device_count(), "GPUs")

def get_module(m):
    return m.module if isinstance(m, nn.DataParallel) else m

## 4. Verification evaluation (EER / AUC on held-out writers)

We embed every val image, sample genuine (same-writer) and impostor (different-writer) pairs, and compute
cosine similarities. EER and the best-accuracy threshold are reported. The chosen threshold is what you ship.

In [ ]:
@torch.no_grad()
def embed_writer_images(model, w2imgs, max_per_writer=40, batch=256):
    '''Return embeddings (N, D) and integer writer ids for a capped set of val images.'''
    model.eval()
    paths, ids = [], []
    for i, (w, imgs) in enumerate(w2imgs.items()):
        pick = imgs if len(imgs) <= max_per_writer else random.Random(SEED).sample(imgs, max_per_writer)
        paths += pick; ids += [i] * len(pick)
    embs = []
    for s in range(0, len(paths), batch):
        xb = torch.stack([make_tensor(fit_image(p), train=False) for p in paths[s:s+batch]]).to(device)
        embs.append(get_module(model).embed(xb).cpu())
    return torch.cat(embs).numpy(), np.array(ids)

def verification_metrics(embs, ids, n_pairs=20000, seed=SEED):
    rng = np.random.default_rng(seed)
    by_id = {}
    for idx, w in enumerate(ids):
        by_id.setdefault(int(w), []).append(idx)
    by_id = {k: v for k, v in by_id.items() if len(v) >= 2}
    keys = list(by_id.keys())

    gen, imp = [], []
    # genuine pairs
    for _ in range(n_pairs):
        k = keys[rng.integers(len(keys))]
        a, b = rng.choice(by_id[k], size=2, replace=False)
        gen.append((a, b))
    # impostor pairs
    for _ in range(n_pairs):
        k1, k2 = rng.choice(keys, size=2, replace=False)
        a = by_id[k1][rng.integers(len(by_id[k1]))]
        b = by_id[k2][rng.integers(len(by_id[k2]))]
        imp.append((a, b))

    def cos(pairs):
        A = embs[[p[0] for p in pairs]]; B = embs[[p[1] for p in pairs]]
        return (A * B).sum(1)   # embeddings already L2-normalized
    s_gen, s_imp = cos(gen), cos(imp)
    scores = np.concatenate([s_gen, s_imp])
    labels = np.concatenate([np.ones_like(s_gen), np.zeros_like(s_imp)])

    auc = roc_auc_score(labels, scores)
    fpr, tpr, thr = roc_curve(labels, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[i] + fnr[i]) / 2
    # best-accuracy threshold
    accs = [( (s_gen >= t).mean() + (s_imp < t).mean() ) / 2 for t in thr]
    bi = int(np.argmax(accs))
    return {"auc": auc, "eer": eer, "eer_thr": float(thr[i]),
            "best_acc": float(accs[bi]), "best_thr": float(thr[bi])}

## 5. Training loop

- Margin ramps `0 -> ARC_M` over the first few epochs (prevents early collapse with only ~370 classes).
- LR: linear warmup then cosine decay.
- Dataset re-samples 60 images/writer at the start of each epoch.
- After each epoch we run verification on **held-out** writers and keep the best EER checkpoint.

In [ ]:
def load_for_resume(path, model, optimizer=None, scaler=None, map_location=device):
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    get_module(model).load_state_dict(state)
    start_epoch, best_eer = 0, 1.0
    if isinstance(ckpt, dict):
        if optimizer is not None and ckpt.get("optimizer") is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        if scaler is not None and ckpt.get("scaler") is not None:
            scaler.load_state_dict(ckpt["scaler"])
        if "epoch" in ckpt:
            start_epoch = int(ckpt["epoch"]) + 1          # resume AFTER the saved epoch
        best_eer = float(ckpt.get("best_eer", ckpt.get("metrics", {}).get("eer", 1.0)))
    print(f"Resumed from {path} | start_epoch={start_epoch} | best_eer={best_eer:.4f}")
    return start_epoch, best_eer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=BASE_LR,
                            momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)
scaler = torch.amp.GradScaler('cuda')

def lr_at(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    p = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * BASE_LR * (1 + math.cos(math.pi * p))

def margin_at(epoch):
    if epoch >= ARC_M_WARMUP_EPOCHS:
        return ARC_M
    return ARC_M * (epoch + 1) / ARC_M_WARMUP_EPOCHS

os.makedirs(OUT_DIR, exist_ok=True)

def save_ckpt(path, epoch, metrics, best_eer):
    '''Full-state checkpoint: weights + optimizer + AMP scaler + bookkeeping (resumable).'''
    torch.save({"model": get_module(model).state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "config": {"LAYERS": LAYERS, "EMBED_DIM": EMBED_DIM,
                           "IMG_H": IMG_H, "IMG_W": IMG_W, "PAD_VALUE": PAD_VALUE},
                "best_thr": metrics["best_thr"], "eer_thr": metrics["eer_thr"],
                "metrics": metrics, "epoch": epoch, "best_eer": best_eer},
               path)

start_epoch, best_eer = 0, 1.0
if RESUME_PATH:
    start_epoch, best_eer = load_for_resume(RESUME_PATH, model, optimizer, scaler)

history = []

for epoch in range(start_epoch, EPOCHS):
    train_ds.resample()                     # <-- new 60 images/writer this epoch
    for g in optimizer.param_groups:
        g["lr"] = lr_at(epoch)
    get_module(model).arc._set_margin(margin_at(epoch))

    model.train()
    t0 = time.time(); run_loss = correct = seen = 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(x, y)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                                    # un-scale grads first
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)       # then clip
        scaler.step(optimizer); scaler.update()
        run_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item(); seen += x.size(0)

    embs, ids = embed_writer_images(get_module(model), val_w2imgs)
    m = verification_metrics(embs, ids)
    history.append({"epoch": epoch, **m})
    print(f"ep {epoch:02d} | lr {lr_at(epoch):.4f} m {margin_at(epoch):.2f} | "
          f"loss {run_loss/seen:.3f} train_acc {correct/seen:.3f} | "
          f"AUC {m['auc']:.4f} EER {m['eer']:.4f} "
          f"acc {m['best_acc']:.4f} thr {m['best_thr']:.3f} | {time.time()-t0:.0f}s")


    if m["eer"] < best_eer:
        best_eer = m["eer"]
        save_ckpt(os.path.join(OUT_DIR, "best_verifier.pt"), epoch, m, best_eer)
        print(f"  >> saved best (EER {best_eer:.4f}, thr {m['best_thr']:.3f})")

   
    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
        save_ckpt(os.path.join(OUT_DIR, "last_verifier.pt"), epoch, m, best_eer)        
        save_ckpt(os.path.join(OUT_DIR, f"ckpt_ep{epoch:02d}.pt"), epoch, m, best_eer)  
        print(f"  >> periodic checkpoint saved (epoch {epoch}) -> last_verifier.pt, ckpt_ep{epoch:02d}.pt")

print("Best EER:", best_eer)